# Structured Contrastive Embedding -- Distinct Semantic Subspaces

`StructuredContrastiveLoss` combines per-semantic SupCon (each chunk
specializes for its own aspect's positives) with a cross-chunk decorrelation
term that drives the cross-correlation between different chunks toward zero
(chunks encode distinct information). A per-chunk variance hinge prevents
collapse.

This notebook reads **pre-split** training and holdout data and per-semantic
cluster labels from disk, runs the training loop with per-epoch holdout
monitoring + best-by-val-loss snapshot, and visualizes each chunk's
clustering geometry on the holdout split.

Expected file layout (point `DATA_DIR` at your data):

```
DATA_DIR/train/    numeric.npy  missing.npy  categorical.npy
                   balance.npy  payment.npy  delinquency.npy  other.npy    <- labels
DATA_DIR/holdout/  numeric.npy  missing.npy  categorical.npy
                   balance.npy  payment.npy  delinquency.npy  other.npy    <- labels
```

Each label file is a `(N_split,)` int64 array of cluster ids for that
semantic. If files are missing, the demo bootstrap cell generates a synthetic
example end-to-end.

In [ ]:
import sys, pathlib
REPO_ROOT = pathlib.Path.cwd()
if not (REPO_ROOT / 'ts_embed').exists():
    REPO_ROOT = REPO_ROOT.parent
sys.path.insert(0, str(REPO_ROOT))

import numpy as np
import torch
from torch.utils.data import Dataset, DataLoader

from ts_embed.data import (DatasetPaths, TimeSeriesDataset,
                           TimeFeatureMasker, ContrastiveCollator)
from ts_embed.model import TSEmbeddingModel, TSEncoderConfig
from ts_embed.loss import StructuredContrastiveLoss, SemanticSpec

torch.manual_seed(0); np.random.seed(0)
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print('device:', device)

## 1. Configuration + data layout

Point `DATA_DIR` at your data directory. The notebook expects the file
layout described in the intro. The demo bootstrap (next cell) creates
synthetic train + holdout data and per-semantic labels there if any of the
expected files are missing -- **skip it if you have your own**.

In [ ]:
# Demo-only constants (used by the bootstrap cell below).
N, T, F_NUM, F_CAT = 6000, 24, 98, 2
K = 3                                       # latent groups per semantic
ASPECT_NAMES = ['balance', 'payment', 'delinquency', 'other']
BLK = {'balance': slice(0, 25), 'payment': slice(25, 50),
       'delinquency': slice(50, 75), 'other': slice(75, 98)}

DATA_DIR    = REPO_ROOT / 'data_demo'
TRAIN_DIR   = DATA_DIR / 'train'
HOLDOUT_DIR = DATA_DIR / 'holdout'


def _files_exist(d):
    needed = ['numeric.npy', 'missing.npy', 'categorical.npy'] + [f'{a}.npy' for a in ASPECT_NAMES]
    return all((d / fn).exists() for fn in needed)


if _files_exist(TRAIN_DIR) and _files_exist(HOLDOUT_DIR):
    print('train + holdout files already exist; skipping demo bootstrap')
else:
    print('generating synthetic train + holdout demo data + labels ...')
    rng = np.random.default_rng(0)

    groups = {nm: rng.integers(0, K, N) for nm in BLK}
    numeric = 0.3 * rng.standard_normal((N, T, F_NUM)).astype(np.float32)
    t_axis = np.linspace(0, 1, T).astype(np.float32)

    lvl = np.array([-1.0, 0.0, 1.2], np.float32)
    slope = np.array([0.0, 0.8, -0.6], np.float32)
    for g in range(K):
        m = groups['balance'] == g
        curve = (lvl[g] + slope[g] * t_axis)[None, :, None]
        numeric[m, :, BLK['balance']] += curve

    pr = np.array([0.2, 0.6, 0.95], np.float32)
    reg = np.array([0.5, 0.2, 0.05], np.float32)
    for g in range(K):
        m = groups['payment'] == g
        numeric[m, :, BLK['payment']] += pr[g] + reg[g] * rng.standard_normal(
            (int(m.sum()), T, 25)).astype(np.float32)

    freq = np.array([0.02, 0.15, 0.45], np.float32)
    for g in range(K):
        m = groups['delinquency'] == g
        bursts = (rng.random((int(m.sum()), T, 25)) < freq[g]).astype(np.float32) * 2.0
        numeric[m, :, BLK['delinquency']] += bursts

    other_tpl = rng.standard_normal((K, 23)).astype(np.float32) * 1.3
    for g in range(K):
        m = groups['other'] == g
        numeric[m, :, BLK['other']] += other_tpl[g][None, None, :]

    missing = (rng.random((N, T, F_NUM)) < 0.1).astype(np.uint8)
    feat_mean = numeric.mean((0, 1), keepdims=True)
    numeric = np.where(missing == 1, feat_mean, numeric).astype(np.float32)

    del_p = np.array([0.05, 0.4, 0.85], np.float32)[groups['delinquency']]
    oth_p = np.array([0.1, 0.5, 0.9], np.float32)[groups['other']]
    categorical = np.stack([
        (rng.random((N, T)) < del_p[:, None]).astype(np.int8),
        (rng.random((N, T)) < oth_p[:, None]).astype(np.int8),
    ], axis=-1)

    # Per-semantic descriptors, then KMeans -> labels. Fit on TRAIN rows and
    # predict on HOLDOUT so the two splits share the same cluster ids.
    t_idx = np.arange(T, dtype=np.float32); t_ctr = t_idx - t_idx.mean()

    def desc(block):
        b = numeric[:, :, block]
        level = b.mean((1, 2))[:, None]
        vol   = b.std(1).mean(1)[:, None]
        sl    = (b * t_ctr[None, :, None]).sum(1) / (t_ctr ** 2).sum()
        trend = sl.mean(1)[:, None]
        return np.concatenate([level, vol, trend], 1)

    descs = {
        'balance':     desc(BLK['balance']),
        'payment':     desc(BLK['payment']),
        'delinquency': np.concatenate([desc(BLK['delinquency']),
                                       categorical[:, :, 0].mean(1)[:, None]], 1),
        'other':       np.concatenate([desc(BLK['other']),
                                       categorical[:, :, 1].mean(1)[:, None]], 1),
    }

    perm = rng.permutation(N)
    train_rows   = perm[:int(0.85 * N)]
    holdout_rows = perm[int(0.85 * N):]

    try:
        from sklearn.cluster import KMeans
        def fit_predict(d):
            km = KMeans(K, n_init=10, random_state=0).fit(d[train_rows])
            return km.labels_, km.predict(d[holdout_rows])
    except ImportError:
        def fit_predict(d):
            tr = d[train_rows]
            C = tr[rng.choice(len(tr), K, replace=False)]
            for _ in range(50):
                lab_tr = ((tr[:, None] - C[None]) ** 2).sum(-1).argmin(1)
                C = np.stack([tr[lab_tr == j].mean(0) if (lab_tr == j).any() else C[j]
                              for j in range(K)])
            lab_ho = ((d[holdout_rows][:, None] - C[None]) ** 2).sum(-1).argmin(1)
            return lab_tr, lab_ho

    train_labels, holdout_labels = {}, {}
    for a in ASPECT_NAMES:
        tr_lab, ho_lab = fit_predict(descs[a])
        train_labels[a]   = tr_lab.astype(np.int64)
        holdout_labels[a] = ho_lab.astype(np.int64)

    def _save(d, rows, labels_dict):
        d.mkdir(parents=True, exist_ok=True)
        np.save(d / 'numeric.npy',     numeric[rows])
        np.save(d / 'missing.npy',     missing[rows])
        np.save(d / 'categorical.npy', categorical[rows])
        for a in ASPECT_NAMES:
            np.save(d / f'{a}.npy', labels_dict[a])

    _save(TRAIN_DIR,   train_rows,   train_labels)
    _save(HOLDOUT_DIR, holdout_rows, holdout_labels)
    print(f'  wrote train ({len(train_rows)}) + holdout ({len(holdout_rows)}) to {DATA_DIR}')

## 2. Load training + holdout datasets and per-semantic labels

Two separate `TimeSeriesDataset` instances (one per split), plus per-semantic
label arrays loaded directly from `<split>/<semantic>.npy`.

In [ ]:
train_paths = DatasetPaths(
    numeric=TRAIN_DIR / 'numeric.npy',
    missing=TRAIN_DIR / 'missing.npy',
    categorical=TRAIN_DIR / 'categorical.npy',
)
holdout_paths = DatasetPaths(
    numeric=HOLDOUT_DIR / 'numeric.npy',
    missing=HOLDOUT_DIR / 'missing.npy',
    categorical=HOLDOUT_DIR / 'categorical.npy',
)
train_base   = TimeSeriesDataset(train_paths)
holdout_base = TimeSeriesDataset(holdout_paths)

train_labels_np   = {a: np.load(TRAIN_DIR   / f'{a}.npy').astype(np.int64) for a in ASPECT_NAMES}
holdout_labels_np = {a: np.load(HOLDOUT_DIR / f'{a}.npy').astype(np.int64) for a in ASPECT_NAMES}
TRAIN_LABELS   = {a: torch.from_numpy(train_labels_np[a])   for a in ASPECT_NAMES}
HOLDOUT_LABELS = {a: torch.from_numpy(holdout_labels_np[a]) for a in ASPECT_NAMES}

print(f'train n={len(train_base)}  |  holdout n={len(holdout_base)}')
for a in ASPECT_NAMES:
    print(f'  {a:12s}: train counts={np.bincount(train_labels_np[a])}  '
          f'holdout counts={np.bincount(holdout_labels_np[a])}')

## 3. Dataset wrapper + label-carrying collator

`TargetDataset` wraps a base dataset and a dict of per-semantic label arrays
so each item carries its labels alongside the feature dict -- no global-index
bookkeeping. The collator stacks them into the batch's `labels` dict, which
`StructuredContrastiveLoss` consumes.

In [ ]:
class TargetDataset(Dataset):
    """Wraps a base dataset and a dict of per-semantic label arrays; each
    item yields the feature dict plus a `_labels` dict of per-aspect
    label scalars."""
    def __init__(self, base, label_dict):
        for k, v in label_dict.items():
            assert len(base) == len(v), f'label `{k}` length {len(v)} != base {len(base)}'
        self.base = base
        self.labels = label_dict

    def __len__(self):
        return len(self.base)

    def __getitem__(self, i):
        item = self.base[i]
        item['_labels'] = {k: v[i] for k, v in self.labels.items()}
        return item


class SemanticCollator:
    def __init__(self, masker):
        self.cc = ContrastiveCollator(masker)

    def __call__(self, batch):
        per_sample = [b.pop('_labels') for b in batch]
        out = self.cc(batch)
        out['labels'] = {
            k: torch.stack([b[k] for b in per_sample]) for k in per_sample[0]
        }
        return out


train_ds   = TargetDataset(train_base,   TRAIN_LABELS)
holdout_ds = TargetDataset(holdout_base, HOLDOUT_LABELS)

masker = TimeFeatureMasker(time_mask_prob=0.25, feature_mask_prob=0.30,
                           n_time_spans=2, feat_span_frac=0.5)
train_loader = DataLoader(train_ds, batch_size=256, shuffle=True, drop_last=True,
                          num_workers=0, collate_fn=SemanticCollator(masker))
val_loader = DataLoader(holdout_ds, batch_size=256, shuffle=False, drop_last=False,
                        num_workers=0, collate_fn=SemanticCollator(masker))
print(f'train batches/epoch: {len(train_loader)}  |  val batches/epoch: {len(val_loader)}')

## 4. Model + structured loss (4 chunks of 48 dims)

`d_model = 192` split into four equal 48-dim chunks. `decorr_weight` controls
how hard distinct chunks are pushed apart; `var_weight` keeps dims active.

In [ ]:
D_MODEL = 192
cfg = TSEncoderConfig(n_numeric=F_NUM, cat_cardinalities=(2, 2), seq_len=T,
                      d_model=D_MODEL, n_layers=2, n_heads=4, proj_dim=128)
model = TSEmbeddingModel(cfg).to(device)

semantics = [
    SemanticSpec('balance',       0,  48, temperature=0.1, contrastive_weight=1.0),
    SemanticSpec('payment',      48,  96, temperature=0.1, contrastive_weight=1.0),
    SemanticSpec('delinquency',  96, 144, temperature=0.1, contrastive_weight=1.0),
    SemanticSpec('other',       144, 192, temperature=0.1, contrastive_weight=1.0),
]
struct_loss = StructuredContrastiveLoss(
    semantics, decorr_weight=1.0, var_weight=1.0, var_gamma=1.0
).to(device)

params = list(model.parameters()) + list(struct_loss.parameters())
optim = torch.optim.AdamW(params, lr=1e-3, weight_decay=1e-4)
print('trainable params:', round(sum(p.numel() for p in params) / 1e6, 2), 'M')

## 5. Training loop with validation + best-model selection

Each epoch:

1. Train through `train_loader`; track running averages of every loss
   component.
2. Run a no-grad pass through `val_loader` with the masking RNG reseeded so
   the val loss is directly comparable across epochs.
3. If `val_loss` improved, snapshot the model + structured-loss state into
   `best_state`.

At the end, save the best checkpoint to `runs/structured/best.pt` and reload
it for evaluation. Watch for:

- **`c_*`** falling on both train and val -- the per-semantic SupCon terms
  are learning.
- **`decorr`** falling -- cross-chunk correlation shrinking (distinct
  subspaces).
- **`val_loss` diverging upward** while `train_loss` keeps falling -- the
  encoder is overfitting the cluster labels.

In [ ]:
EPOCHS = 12
history = []
best_val_loss = float('inf')
best_state = None

@torch.no_grad()
def run_val():
    """Mean structured-loss components on the validation split. Reseeds the
    RNG so the random masking is identical across epochs."""
    model.eval(); struct_loss.eval()
    rng_state = torch.get_rng_state(); torch.manual_seed(123)
    sums = {}; nb_local = 0
    try:
        for batch in val_loader:
            num_a = batch['numeric_a'].to(device); mis_a = batch['missing_a'].to(device)
            keep_a = batch['time_keep_a'].to(device)
            num_b = batch['numeric_b'].to(device); mis_b = batch['missing_b'].to(device)
            keep_b = batch['time_keep_b'].to(device)
            cat_a = batch['categorical_a'].to(device)
            cat_b = batch['categorical_b'].to(device)
            labels = {k: v.to(device) for k, v in batch['labels'].items()}
            emb_a = model.encode(num_a, mis_a, cat_a, keep_a)
            emb_b = model.encode(num_b, mis_b, cat_b, keep_b)
            out = struct_loss(emb_a, emb_b, labels=labels)
            for k, v in out.items():
                sums[k] = sums.get(k, 0.0) + float(v)
            nb_local += 1
    finally:
        torch.set_rng_state(rng_state)
    return {k: v / max(nb_local, 1) for k, v in sums.items()}


for epoch in range(EPOCHS):
    model.train(); struct_loss.train()
    train_sums = {}; nb_local = 0
    for batch in train_loader:
        num_a = batch['numeric_a'].to(device); mis_a = batch['missing_a'].to(device)
        keep_a = batch['time_keep_a'].to(device)
        num_b = batch['numeric_b'].to(device); mis_b = batch['missing_b'].to(device)
        keep_b = batch['time_keep_b'].to(device)
        cat_a = batch['categorical_a'].to(device)
        cat_b = batch['categorical_b'].to(device)
        labels = {k: v.to(device) for k, v in batch['labels'].items()}

        emb_a = model.encode(num_a, mis_a, cat_a, keep_a)
        emb_b = model.encode(num_b, mis_b, cat_b, keep_b)
        out = struct_loss(emb_a, emb_b, labels=labels)

        optim.zero_grad(set_to_none=True)
        out['loss'].backward()
        torch.nn.utils.clip_grad_norm_(params, 1.0)
        optim.step()
        for k, v in out.items():
            train_sums[k] = train_sums.get(k, 0.0) + float(v)
        nb_local += 1

    train_avg = {k: v / max(nb_local, 1) for k, v in train_sums.items()}
    val_avg = run_val()
    history.append({'epoch': epoch, 'train': train_avg, 'val': val_avg})

    improved = val_avg['loss'] < best_val_loss
    if improved:
        best_val_loss = val_avg['loss']
        best_state = {
            'model':       {k: v.detach().cpu().clone() for k, v in model.state_dict().items()},
            'struct_loss': {k: v.detach().cpu().clone() for k, v in struct_loss.state_dict().items()},
            'cfg': cfg.__dict__,
            'epoch': epoch,
            'val_loss': val_avg['loss'],
            'val_components': {k: v for k, v in val_avg.items() if k != 'loss'},
        }

    print(f"epoch {epoch:2d}  train={train_avg['loss']:.3f}  val={val_avg['loss']:.3f}  "
          f"(val: c_bal={val_avg['c_balance']:.3f} c_pay={val_avg['c_payment']:.3f} "
          f"c_del={val_avg['c_delinquency']:.3f} c_oth={val_avg['c_other']:.3f}  "
          f"decorr={val_avg['decorr']:.4f})"
          + ('  <- best' if improved else ''))

run_dir = REPO_ROOT / 'runs' / 'structured'
run_dir.mkdir(parents=True, exist_ok=True)
torch.save(best_state, run_dir / 'best.pt')
print(f'\nbest val_loss = {best_val_loss:.3f}  @ epoch {best_state["epoch"]}  ->  {run_dir / "best.pt"}')

## 6. Reload best model + diagnostics on HOLDOUT

Reload the best-by-`val_loss` checkpoint, extract embeddings for every
**holdout** account, and run two diagnostics:

1. **Specialization** -- kNN-classify every (chunk, semantic-label) pair on
   the holdout split. The 4x4 accuracy matrix should be **high on the
   diagonal**, near chance off it.
2. **Disentanglement** -- mean |cross-correlation| between chunks should be
   **near zero off-diagonal**.

Plus a per-chunk std collapse guard. We evaluate on holdout so the diagnostics
aren't biased by labels the encoder saw during training.

In [ ]:
# Reload best-validated checkpoint into the live model.
model.load_state_dict(best_state['model'])
struct_loss.load_state_dict(best_state['struct_loss'])


@torch.no_grad()
def embed_dataset(ts_dataset):
    model.eval()
    n_rows = int(len(ts_dataset))
    Z = []
    for start in range(0, n_rows, 512):
        end = start + 512 if start + 512 < n_rows else n_rows
        sl = slice(start, end)
        num = torch.from_numpy(np.asarray(ts_dataset.numeric[sl], dtype=np.float32)).to(device)
        mis = torch.from_numpy(np.asarray(ts_dataset.missing[sl], dtype=np.float32)).to(device)
        cat = torch.from_numpy(np.asarray(ts_dataset.categorical[sl], dtype=np.int64)).to(device)
        Z.append(model.encode(num, mis, cat).float().cpu().numpy())
    return np.concatenate(Z)


# All downstream diagnostics + viz use HOLDOUT embeddings + HOLDOUT labels.
embs = embed_dataset(holdout_base)
EVAL_LABELS_NP = holdout_labels_np

CH = {'balance': slice(0, 48), 'payment': slice(48, 96),
      'delinquency': slice(96, 144), 'other': slice(144, 192)}


def knn_acc(X, y, k=15, seed=0):
    rng_local = np.random.default_rng(seed)
    perm = rng_local.permutation(len(X))
    cut = int(0.7 * len(X)); tr, te = perm[:cut], perm[cut:]
    Xt = X[tr] / (np.linalg.norm(X[tr], axis=1, keepdims=True) + 1e-8)
    Xe = X[te] / (np.linalg.norm(X[te], axis=1, keepdims=True) + 1e-8)
    nn = np.argpartition(-(Xe @ Xt.T), k, axis=1)[:, :k]
    pred = np.array([np.bincount(r).argmax() for r in y[tr][nn]])
    return float((pred == y[te]).mean())


names = list(CH)
K_clusters = max(int(EVAL_LABELS_NP[a].max()) for a in names) + 1

print(f'kNN accuracy on HOLDOUT  (rows=chunk, cols=semantic label)  chance={1.0 / K_clusters:.2f}')
print('              ' + '  '.join(f'{n:>11s}' for n in names))
knn_mat = np.zeros((len(names), len(names)))
for ci, cn in enumerate(names):
    for li, ln in enumerate(names):
        knn_mat[ci, li] = knn_acc(embs[:, CH[cn]], EVAL_LABELS_NP[ln])
    print(f'{cn:12s}  ' + '  '.join(f'{v:11.3f}' for v in knn_mat[ci]))

Z = {cn: (embs[:, CH[cn]] - embs[:, CH[cn]].mean(0)) /
         (embs[:, CH[cn]].std(0) + 1e-8) for cn in names}
print('\nmean |cross-correlation| between chunks on HOLDOUT:')
print('              ' + '  '.join(f'{n:>11s}' for n in names))
corr_mat = np.zeros((len(names), len(names)))
for ai, a in enumerate(names):
    for bi, b in enumerate(names):
        corr_mat[ai, bi] = float(np.abs(Z[a].T @ Z[b] / len(embs)).mean())
    print(f'{a:12s}  ' + '  '.join(f'{v:11.3f}' for v in corr_mat[ai]))

for cn in names:
    s = embs[:, CH[cn]].std(0)
    assert s.mean() > 1e-2, f'COLLAPSE in {cn} chunk'
print('\nExpect: kNN diagonal high / off-diagonal ~chance, and cross-correlation'
      ' off-diagonal ~0 => distinct semantic subspaces.')

## 7. Visualization

Three plots:

1. **Training curves** -- train vs val loss across epochs, with the
   best-val epoch annotated.
2. **kNN-accuracy and cross-correlation heatmaps** -- summary view of
   specialization (diagonal-heavy) and disentanglement (off-diagonal ~0).
3. **Per-chunk PCA scatter grid** -- 4 x 4 panel. Each row is one chunk
   projected to 2D via PCA; each column colors the points by one of the four
   semantic labels. The **diagonal** (green border) should show clear
   clusters (chunk specialized for its own aspect); **off-diagonal** should
   look diffuse (the chunk doesn't encode the other aspect).

In [ ]:
try:
    import matplotlib.pyplot as plt

    epochs = [h['epoch'] for h in history]
    tr = [h['train']['loss'] for h in history]
    va = [h['val']['loss']   for h in history]

    fig, axes = plt.subplots(1, 3, figsize=(16, 4.5))

    axes[0].plot(epochs, tr, '-o', label='train', linewidth=2)
    axes[0].plot(epochs, va, '-s', label='val',   linewidth=2)
    axes[0].axvline(best_state['epoch'], color='k', linestyle=':', alpha=0.6,
                     label=f"best epoch {best_state['epoch']}")
    axes[0].set_xlabel('epoch'); axes[0].set_ylabel('total loss')
    axes[0].set_title('train / val loss'); axes[0].legend(); axes[0].grid(alpha=0.3)

    im1 = axes[1].imshow(knn_mat, vmin=0, vmax=1, cmap='viridis')
    axes[1].set_xticks(range(len(names))); axes[1].set_xticklabels(names, rotation=30, ha='right')
    axes[1].set_yticks(range(len(names))); axes[1].set_yticklabels(names)
    axes[1].set_xlabel('label aspect'); axes[1].set_ylabel('chunk')
    axes[1].set_title(f'kNN accuracy on holdout  (chance={1.0 / K_clusters:.2f})')
    for i in range(len(names)):
        for j in range(len(names)):
            axes[1].text(j, i, f'{knn_mat[i, j]:.2f}', ha='center', va='center',
                         color='white' if knn_mat[i, j] < 0.55 else 'black', fontsize=9)
    plt.colorbar(im1, ax=axes[1], fraction=0.046)

    im2 = axes[2].imshow(corr_mat, vmin=0, vmax=max(corr_mat.max(), 0.05),
                         cmap='magma')
    axes[2].set_xticks(range(len(names))); axes[2].set_xticklabels(names, rotation=30, ha='right')
    axes[2].set_yticks(range(len(names))); axes[2].set_yticklabels(names)
    axes[2].set_title('mean |cross-correlation| between chunks (holdout)')
    for i in range(len(names)):
        for j in range(len(names)):
            axes[2].text(j, i, f'{corr_mat[i, j]:.2f}', ha='center', va='center',
                         color='white' if corr_mat[i, j] < 0.5 * corr_mat.max() else 'black', fontsize=9)
    plt.colorbar(im2, ax=axes[2], fraction=0.046)

    plt.tight_layout(); plt.show()
except ImportError:
    print('matplotlib not installed; skipping training-curve and heatmap plots')

In [ ]:
try:
    import matplotlib.pyplot as plt

    def pca2d(X, seed=0):
        try:
            from sklearn.decomposition import PCA
            return PCA(n_components=2, random_state=seed).fit_transform(X)
        except ImportError:
            Xc = X - X.mean(0)
            _, _, Vt = np.linalg.svd(Xc, full_matrices=False)
            return Xc @ Vt[:2].T

    rng_local = np.random.default_rng(0)
    n_show = min(2500, len(embs))
    sample_idx = rng_local.choice(len(embs), size=n_show, replace=False)

    n_chunks = len(names)
    fig, axes = plt.subplots(n_chunks, n_chunks, figsize=(14, 14))
    for i, chunk_name in enumerate(names):
        Z2 = pca2d(embs[sample_idx][:, CH[chunk_name]])
        for j, label_name in enumerate(names):
            ax = axes[i, j]
            colors = EVAL_LABELS_NP[label_name][sample_idx]
            ax.scatter(Z2[:, 0], Z2[:, 1], c=colors, cmap='tab10', s=6, alpha=0.65)
            if i == 0:
                ax.set_title(f'colored by\n{label_name}', fontsize=10)
            if j == 0:
                ax.set_ylabel(f'{chunk_name} chunk\n(PCA 2D, holdout)', fontsize=10)
            ax.set_xticks([]); ax.set_yticks([])
            if i == j:
                for spine in ax.spines.values():
                    spine.set_edgecolor('green'); spine.set_linewidth(2.5)
    fig.suptitle("Each chunk's 2D PCA on HOLDOUT, colored by every semantic label.  "
                 "Diagonal (green) = chunk vs OWN label (should show CLUSTERS); "
                 "off-diagonal = chunk vs OTHER (should be DIFFUSE).",
                 fontsize=11, y=1.00)
    plt.tight_layout(rect=[0, 0, 1, 0.97]); plt.show()
except ImportError:
    print('matplotlib not installed; skipping scatter grid')

## Notes / tuning

- **Real positives**: swap KMeans labels for production descriptors, or pass
  `pos_masks={'balance': BxB bool, ...}` to use a descriptor-similarity graph.
  With no labels for a semantic, that chunk falls back to self-supervised
  (only the two augmented views are positives) and still gets decorrelated.
- **`decorr_weight`**: raise it if the cross-correlation off-diagonal stays
  high (chunks redundant); lower it if specialization (diagonal kNN) suffers
  because the constraint is too aggressive.
- **`var_weight`**: raise if a chunk's per-dim std drifts toward 0.
- **Chunk sizing**: chunks need not be equal — give noisier semantics fewer
  dims. Keep slices disjoint and covering `[0, d_model)`.
- **Scale-out**: wire `labels` through the collator exactly as here and reuse
  `scripts/train_ddp.py`; for very wide batches gather features across ranks
  (mirror VICReg's `_maybe_all_gather`).